# 01 Build the slice cache (CPU, no GPU needed)
1. Session options: Accelerator **None**, Internet **ON**, secret `GITHUB_TOKEN` attached.
2. **+ Add Input** -> your raw Dataset (`mnj-raw-ispy1`, `mnj-raw-ispy2` or `mnj-raw-duke`).
3. Set the parameters, then **Save Version -> Save & Run All (Commit)**.
4. When it finishes: check the QC numbers and the grid below, then **New Dataset** from the output,
   named `mnj-cache-<cohort>` (Private, shared with the team). Post the status counts in the chat.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "OWNER/REPO"
GIT_TAG = "d1"          # code version; A posts the tag to use
PERSON = "B"            # "A", "B" or "C"
COHORT = "ispy1"       # "ispy1", "ispy2" or "duke"
LIMIT = 0               # 0 = all patients; 10 for a quick check
RAW = ""                # leave "" to find the mnj-raw-<cohort> input automatically

In [ ]:
# ==== SETUP (do not edit) ====
import base64, json, os, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
_hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    r = subprocess.run(["git", "-c", f"http.extraHeader={_hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
os.chdir("/tmp/repo")
print("code at", GIT_TAG, git("log", "-1", "--format=%h %s"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import glob
if not RAW:
    hits = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
            if f"raw-{COHORT}" in d.rstrip("/").split("/")[-1]]
    assert hits, f"attach the mnj-raw-{COHORT} Dataset (+ Add Input)"
    RAW = hits[0]
OUT = f"/kaggle/working/cache_{COHORT}"
print("raw:", RAW, os.listdir(RAW))
cmd = [sys.executable, "scripts/build_cache.py", "--cohort", COHORT, "--raw", RAW, "--out", OUT, "--workers", "4"]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display
idx = pd.read_csv(f"{OUT}/cache_index.csv")
print(json.load(open(f"{OUT}/cache_meta.json")))
print(idx.status.value_counts())
if "z_source" in idx:
    print(idx.z_source.value_counts(dropna=False))
if "mask_csv_iou_yx" in idx:
    print("mask vs CSV box IoU:", idx.mask_csv_iou_yx.describe().round(3).to_dict())
display(idx[idx.status == "error"].head(20))
display(Image(f"{OUT}/summary_grid.png"))